## Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd

from darts.models import Chronos2Model

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data import load_dataset

from src.advanced import (
    dataframe_to_multiple_series,
    chronological_split_multiple_series,
    check_series_alignment,
    summarize_multiple_series,
)

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


## Load dataset

In [2]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "LD2011_2014.txt"
)

df = load_dataset(
    str(DATA_PATH)
)

print(
    "Dataset shape:",
    df.shape
)

display(
    df.head()
)

Loading dataset: c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\data\raw\LD2011_2014.txt


c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\src\data.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.reset_index()


Raw shape: (140256, 371)
Dataset shape: (51894720, 3)


,timestamp,series_id,value
0,2011-01-01 00:15:00,MT_001,0.0
1,2011-01-01 00:30:00,MT_001,0.0
2,2011-01-01 00:45:00,MT_001,0.0
3,2011-01-01 01:00:00,MT_001,0.0
4,2011-01-01 01:15:00,MT_001,0.0


In [4]:
from src.data import load_raw_dataset  # wide format, no melt applied

raw_df = load_raw_dataset(DATA_PATH)  # this already has MT_001, MT_002... as real columns

series_dict = dataframe_to_multiple_series(
    raw_df,
    series_ids=SERIES_IDS,
)

Loading dataset: c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\data\raw\LD2011_2014.txt


c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\src\data.py:70: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = df.reset_index()


Raw shape: (140256, 371)


## Select all Series

In [16]:
from src.advanced import get_series_columns

SERIES_IDS = get_series_columns(raw_df)  # all 370 series, timestamp excluded
print(f"Total series: {len(SERIES_IDS)}")

series_dict = dataframe_to_multiple_series(
    raw_df,
    series_ids=SERIES_IDS,
)

Total series: 370


In [17]:
display(
    check_series_alignment(
        series_dict
    )
)

,series_id,length,start,end,frequency
0,MT_001,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
1,MT_002,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
2,MT_003,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
3,MT_004,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
4,MT_005,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
...,...,...,...,...,...
365,MT_366,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
366,MT_367,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
367,MT_368,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>
368,MT_369,140256,2011-01-01 00:15:00,2015-01-01,<15 * Minutes>


In [18]:
display(
    summarize_multiple_series(
        series_dict
    )
)

,Series_ID,Observations,Minimum,Maximum,Mean,Std,Missing
0,MT_001,140256,0.0,48.223350,3.970785,5.983944,0
1,MT_002,140256,0.0,115.220484,20.768480,13.272368,0
2,MT_003,140256,0.0,151.172893,2.918308,11.014417,0
3,MT_004,140256,0.0,321.138211,82.184490,58.248184,0
4,MT_005,140256,0.0,150.000000,37.240309,26.461233,0
...,...,...,...,...,...,...,...
365,MT_366,140256,0.0,60.269163,9.269709,10.016746,0
366,MT_367,140256,0.0,1138.718174,424.262904,274.336144,0
367,MT_368,140256,0.0,362.270451,94.704717,80.297015,0
368,MT_369,140256,0.0,1549.120235,625.251734,380.654685,0


In [19]:
train_dict, test_dict = (
    chronological_split_multiple_series(
        series_dict,
        train_ratio=0.8,
    )
)

## Chronos-2 Multiple Series

In [20]:
HORIZON = 96

model = Chronos2Model(
    input_chunk_length=512,
    output_chunk_length=HORIZON,
)

model.fit(
    list(train_dict.values())
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Chronos2Model(output_chunk_shift=0, likelihood=None, hub_model_name=amazon/chronos-2, hub_model_revision=None, local_dir=None, input_chunk_length=512, output_chunk_length=96)

In [21]:
forecasts = model.predict(
    n=HORIZON,
    series=list(train_dict.values()),
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

## Evaluate Every Series

In [22]:
from darts.metrics import mae, rmse

results = []

for series_id, forecast in zip(
    train_dict.keys(),
    forecasts,
):

    actual = test_dict[series_id]

    results.append(
        {
            "Model": "Chronos-2",
            "Series_ID": series_id,
            "Horizon": HORIZON,
            "MAE": mae(
                actual,
                forecast,
            ),
            "RMSE": rmse(
                actual,
                forecast,
            ),
        }
    )

multiseries_results = pd.DataFrame(
    results
)

display(
    multiseries_results
)

,Model,Series_ID,Horizon,MAE,RMSE
0,Chronos-2,MT_001,96,0.763720,1.182067
1,Chronos-2,MT_002,96,2.076384,2.801993
2,Chronos-2,MT_003,96,0.156924,0.443147
3,Chronos-2,MT_004,96,10.451502,13.288156
4,Chronos-2,MT_005,96,10.497521,12.678935
...,...,...,...,...,...
365,Chronos-2,MT_366,96,2.833250,4.394834
366,Chronos-2,MT_367,96,35.618807,47.426827
367,Chronos-2,MT_368,96,15.067965,19.800599
368,Chronos-2,MT_369,96,36.733259,46.682129


## Aggregate results

In [23]:
summary = (
    multiseries_results
    .groupby("Model")
    [["MAE", "RMSE"]]
    .mean()
    .reset_index()
)

display(summary)

,Model,MAE,RMSE
0,Chronos-2,44.35095,59.973708


In [24]:
per_series = (
    multiseries_results
    .sort_values("RMSE")
)

display(
    per_series
)

,Model,Series_ID,Horizon,MAE,RMSE
222,Chronos-2,MT_223,96,8.870238e-10,1.181831e-09
177,Chronos-2,MT_178,96,8.870238e-10,1.181831e-09
65,Chronos-2,MT_066,96,1.526920e-01,3.230660e-01
2,Chronos-2,MT_003,96,1.569239e-01,4.431471e-01
149,Chronos-2,MT_150,96,3.160740e-01,4.511412e-01
...,...,...,...,...,...
156,Chronos-2,MT_157,96,3.522110e+02,4.714694e+02
195,Chronos-2,MT_196,96,8.461206e+02,1.090228e+03
278,Chronos-2,MT_279,96,8.588846e+02,1.301722e+03
369,Chronos-2,MT_370,96,1.210274e+03,1.424526e+03


## Save results

In [25]:
RESULTS_DIR = (
    PROJECT_ROOT / "results"
)

multiseries_results.to_csv(
    RESULTS_DIR / "multiseries_metrics.csv",
    index=False,
)

## Imports and Target Selection

In [26]:
from darts.models import Chronos2Model
from darts.metrics import mae, rmse

from src.advanced import (
    create_cyclic_time_covariates,
    create_future_cyclic_covariates,
)

TARGET_SERIES_ID = "MT_001"
HORIZON = 96

target = series_dict[TARGET_SERIES_ID]
train_target, test_target = target.split_before(0.8)

print(f"Train length: {len(train_target)}")
print(f"Test length: {len(test_target)}")

Train length: 112204
Test length: 28052


## Building covariates

In [27]:
# Covers the full training window PLUS the forecast horizon,
# since future covariates must be known ahead of time for predict()
covariates = create_future_cyclic_covariates(
    train_target,
    forecast_horizon=HORIZON,
)

display(covariates.pd_dataframe().head() if hasattr(covariates, "pd_dataframe") else covariates.to_dataframe().head())

,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
timestamp,,,,,,
2011-01-01 00:15:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 00:30:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 00:45:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:00:00,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:15:00,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0


In [28]:
model_b1 = Chronos2Model(
    input_chunk_length=512,
    output_chunk_length=HORIZON,
)

model_b1.fit(train_target)

forecast_b1 = model_b1.predict(
    n=HORIZON,
    series=train_target,
)

actual_b1 = test_target[:HORIZON]

metrics_b1 = {
    "Experiment": "B1 - Target Only",
    "Series_ID": TARGET_SERIES_ID,
    "Horizon": HORIZON,
    "MAE": mae(actual_b1, forecast_b1),
    "RMSE": rmse(actual_b1, forecast_b1),
}

print(metrics_b1)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

{'Experiment': 'B1 - Target Only', 'Series_ID': 'MT_001', 'Horizon': 96, 'MAE': np.float64(0.763719958059378), 'RMSE': np.float64(1.182066753110058)}


## Target + Cyclic covariates

In [29]:
model_b2 = Chronos2Model(
    input_chunk_length=512,
    output_chunk_length=HORIZON,
)

model_b2.fit(
    train_target,
    future_covariates=covariates,
)

forecast_b2 = model_b2.predict(
    n=HORIZON,
    series=train_target,
    future_covariates=covariates,
)

actual_b2 = test_target[:HORIZON]

metrics_b2 = {
    "Experiment": "B2 - Target + Covariates",
    "Series_ID": TARGET_SERIES_ID,
    "Horizon": HORIZON,
    "MAE": mae(actual_b2, forecast_b2),
    "RMSE": rmse(actual_b2, forecast_b2),
}

print(metrics_b2)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

{'Experiment': 'B2 - Target + Covariates', 'Series_ID': 'MT_001', 'Horizon': 96, 'MAE': np.float64(0.6964105188770965), 'RMSE': np.float64(0.9620210220501219)}


## Comparing

In [30]:
covariate_ablation = pd.DataFrame([metrics_b1, metrics_b2])

covariate_ablation["Improvement_vs_B1_%"] = (
    (covariate_ablation.loc[0, "RMSE"] - covariate_ablation["RMSE"])
    / covariate_ablation.loc[0, "RMSE"]
    * 100
)

display(covariate_ablation)

,Experiment,Series_ID,Horizon,MAE,RMSE,Improvement_vs_B1_%
0,B1 - Target Only,MT_001,96,0.763720,1.182067,0.000000
1,B2 - Target + Covariates,MT_001,96,0.696411,0.962021,18.615339


## Save results

In [31]:
covariate_ablation.to_csv(
    RESULTS_DIR / "covariate_ablation_metrics.csv",
    index=False,
)

In [32]:
from darts.models import Chronos2Model
from darts.metrics import mae, rmse

from src.advanced import (
    create_cyclic_time_covariates,
    create_future_cyclic_covariates,
)

target = series_dict["MT_001"]

## Building future covariates

In [33]:
future_covariates = (
    create_future_cyclic_covariates(
        target,
        forecast_horizon=96,
    )
)

future_covariates

,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
timestamp,,,,,,
2011-01-01 00:15:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 00:30:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 00:45:00,0.000000,1.000000,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:00:00,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
2011-01-01 01:15:00,0.258819,0.965926,-0.974928,-0.222521,0.0,1.0
...,...,...,...,...,...,...
2015-01-01 23:00:00,-0.258819,0.965926,0.433884,-0.900969,0.0,1.0
2015-01-01 23:15:00,-0.258819,0.965926,0.433884,-0.900969,0.0,1.0
2015-01-01 23:30:00,-0.258819,0.965926,0.433884,-0.900969,0.0,1.0


## Split target and covariates

In [34]:
train_target, test_target = (
    target.split_before(0.8)
)

train_covariates = (
    future_covariates
    .slice_intersect(
        train_target
    )
)

## Chronos with covariates

In [35]:
cov_model = Chronos2Model(
    input_chunk_length=512,
    output_chunk_length=96,
)

cov_model.fit(
    train_target,
    future_covariates=train_covariates,
)

cov_forecast = cov_model.predict(
    n=96,
    series=train_target,
    future_covariates=future_covariates,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

##  target-only baseline, for the ablation comparison

In [36]:
baseline_model = Chronos2Model(
    input_chunk_length=512,
    output_chunk_length=96,
)

baseline_model.fit(train_target)

baseline_forecast = baseline_model.predict(
    n=96,
    series=train_target,
)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

## Evaluate and compare

In [37]:
actual = test_target[:96]

metrics_b1 = {
    "Experiment": "B1 - Target Only",
    "Series_ID": "MT_001",
    "Horizon": 96,
    "MAE": mae(actual, baseline_forecast),
    "RMSE": rmse(actual, baseline_forecast),
}

metrics_b2 = {
    "Experiment": "B2 - Target + Cyclic Covariates",
    "Series_ID": "MT_001",
    "Horizon": 96,
    "MAE": mae(actual, cov_forecast),
    "RMSE": rmse(actual, cov_forecast),
}

covariate_ablation = pd.DataFrame([metrics_b1, metrics_b2])

covariate_ablation["Improvement_vs_B1_%"] = (
    (covariate_ablation.loc[0, "RMSE"] - covariate_ablation["RMSE"])
    / covariate_ablation.loc[0, "RMSE"]
    * 100
)

display(covariate_ablation)

,Experiment,Series_ID,Horizon,MAE,RMSE,Improvement_vs_B1_%
0,B1 - Target Only,MT_001,96,0.763720,1.182067,0.000000
1,B2 - Target + Cyclic Covariates,MT_001,96,0.696411,0.962021,18.615339


## save results

In [38]:
covariate_ablation.to_csv(
    RESULTS_DIR / "covariate_ablation_metrics.csv",
    index=False,
)

## Comparing against baseline

In [39]:
from darts.metrics import mae, rmse

actual = test_target[:96]  # match forecast length — same class of bug as before

comparison = pd.DataFrame(
    [
        {
            "Experiment": "Chronos-2",
            "Covariates": "No",
            "MAE": mae(actual, baseline_forecast),
            "RMSE": rmse(actual, baseline_forecast),
        },
        {
            "Experiment": "Chronos-2",
            "Covariates": "Time",
            "MAE": mae(actual, cov_forecast),
            "RMSE": rmse(actual, cov_forecast),
        },
    ]
)

display(comparison)

,Experiment,Covariates,MAE,RMSE
0,Chronos-2,No,0.763720,1.182067
1,Chronos-2,Time,0.696411,0.962021


## Covariate improvement

In [40]:
baseline_rmse = comparison.loc[
    comparison["Covariates"] == "No", "RMSE"
].iloc[0]

covariate_rmse = comparison.loc[
    comparison["Covariates"] == "Time", "RMSE"
].iloc[0]

improvement = (baseline_rmse - covariate_rmse) / baseline_rmse * 100

print(f"RMSE improvement: {improvement:.2f}%")

RMSE improvement: 18.62%


## Experiment Matrix

In [41]:
def run_single_series_experiment(
    target,
    horizon,
    covariates=None,
    input_chunk_length=512,
):
    """
    Fit + forecast Chronos-2 on one series, with or without
    future covariates. Returns a metrics dict.
    """
    train, test = target.split_before(0.8)

    model_kwargs = dict(
        input_chunk_length=input_chunk_length,
        output_chunk_length=horizon,
    )
    model = Chronos2Model(**model_kwargs)

    fit_kwargs = {}
    predict_kwargs = {"n": horizon, "series": train}

    if covariates is not None:
        train_cov = covariates.slice_intersect(train)
        fit_kwargs["future_covariates"] = train_cov
        predict_kwargs["future_covariates"] = covariates

    model.fit(train, **fit_kwargs)
    forecast = model.predict(**predict_kwargs)

    actual = test[:horizon]

    return {
        "MAE": mae(actual, forecast),
        "RMSE": rmse(actual, forecast),
    }


def run_multiseries_experiment(raw_df, n_series, horizon=96):
    """
    Fit + forecast Chronos-2 across n_series series jointly.
    Returns a per-series metrics DataFrame.
    """
    all_ids = get_series_columns(raw_df)
    selected_ids = all_ids[:n_series]

    series_dict = dataframe_to_multiple_series(raw_df, series_ids=selected_ids)
    train_dict, test_dict = chronological_split_multiple_series(series_dict, train_ratio=0.8)

    model = Chronos2Model(input_chunk_length=512, output_chunk_length=horizon)
    model.fit(list(train_dict.values()))

    forecasts = model.predict(n=horizon, series=list(train_dict.values()))

    rows = []
    for series_id, forecast in zip(train_dict.keys(), forecasts):
        actual = test_dict[series_id][:horizon]
        rows.append({
            "N_Series": n_series,
            "Series_ID": series_id,
            "Horizon": horizon,
            "MAE": mae(actual, forecast),
            "RMSE": rmse(actual, forecast),
        })

    return pd.DataFrame(rows)

## Scale Multiple Series

In [42]:
scale_results = []

for n in [10, 50, len(get_series_columns(raw_df))]:
    print(f"Running multi-series experiment with {n} series...")
    result_df = run_multiseries_experiment(raw_df, n_series=n)
    scale_results.append(result_df)

experiment_3_results = pd.concat(scale_results, ignore_index=True)

experiment_3_summary = (
    experiment_3_results
    .groupby("N_Series")[["MAE", "RMSE"]]
    .mean()
    .reset_index()
)

display(experiment_3_summary)

experiment_3_results.to_csv(RESULTS_DIR / "experiment3_scale_metrics.csv", index=False)

Running multi-series experiment with 10 series...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

Running multi-series experiment with 50 series...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

Running multi-series experiment with 370 series...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

,N_Series,MAE,RMSE
0,10,7.200637,8.982418
1,50,13.319243,17.206643
2,370,44.350950,59.973708


## Multiple horizons × covariates, on MT_001

In [43]:
target = series_dict["MT_001"] if "MT_001" in series_dict else dataframe_to_multiple_series(raw_df, series_ids=["MT_001"])["MT_001"]

horizons = {"24h": 96, "48h": 192, "7d": 672}
experiment_5_rows = []

for label, horizon in horizons.items():

    future_covariates = create_future_cyclic_covariates(target, forecast_horizon=horizon)

    for covariate_label, cov in [("Target only", None), ("Time covariates", future_covariates)]:
        print(f"Running {label} / {covariate_label}...")
        metrics = run_single_series_experiment(target, horizon, covariates=cov)
        experiment_5_rows.append({
            "Horizon_Label": label,
            "Horizon_Steps": horizon,
            "Covariates": covariate_label,
            **metrics,
        })

experiment_5_results = pd.DataFrame(experiment_5_rows)
display(experiment_5_results)

experiment_5_results.to_csv(RESULTS_DIR / "experiment5_horizon_covariate_matrix.csv", index=False)

Running 24h / Target only...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

Running 24h / Time covariates...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Predicting: |          | 0/? [00:00<?, ?it/s]

Running 7d / Time covariates...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\palla\OneDrive\Desktop\nita-sem-3\ak-pu\chronos-2\chronos\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Predicting: |          | 0/? [00:00<?, ?it/s]

,Horizon_Label,Horizon_Steps,Covariates,MAE,RMSE
0,24h,96,Target only,0.763720,1.182067
1,24h,96,Time covariates,0.696411,0.962021
2,48h,192,Target only,0.795465,1.208076
3,48h,192,Time covariates,0.727182,0.972690
4,7d,672,Target only,0.964219,1.799736
5,7d,672,Time covariates,0.957010,1.805865
